In [ ]:
import os
import pandas as pd
from torchvision.io import read_image
from torch.utils.data import Dataset
import os
import torch
from torch.utils.data import Dataset
import torchvision
import torchvision.transforms as transforms
import pandas as pd
from vit import ViTDecoder as vit
from PIL import Image
import math as math
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models
import numpy as np

class CustomImageDataset(Dataset):
    def __init__(self, annotations_file, img_dir, transform=None, target_transform=None):
        self.img_labels = pd.read_csv(annotations_file)
        self.img_dir = img_dir
        self.transform = transform
        self.target_transform = target_transform

    def __len__(self):
        return len(self.img_labels)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.img_labels.iloc[idx, 0])
        image = read_image(img_path)
        label = self.img_labels.iloc[idx, 1]
        if self.transform:
            image = self.transform(image)
        if self.target_transform:
            label = self.target_transform(label)
        return image, label

In [ ]:
templist = []
for file in os.listdir("/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input"):
    templist.append(file)

In [ ]:
df = pd.DataFrame(templist)
print(df.sort_values(by=0))

In [ ]:
# Custom Padding Transformer (FIX THIS)
class PadToSize:
    def __init__(self, target):
        self.target = target

    def __call__(self, img):
        # Calculate padding sizes

        colors, width, height = img.size()
        pad_left = math.floor((max(0, self.target[0] - width)/2))
        pad_right = math.ceil(((max(0, self.target[0]-width))/2))-1
        pad_top = math.floor((max(0, self.target[1] - height)/2))+1
        pad_bot = math.ceil((max(0,self.target[1] - height)/2))

        padding = (pad_left, pad_top, pad_right, pad_bot)
        if self.target[0]-width > width or self.target[1]-height > height:
            return transforms.functional.pad(img, padding, fill=0, padding_mode='constant')
        else:
            return transforms.functional.pad(img, padding, fill=0, padding_mode='reflect')
        
class ConvertToFloat32(object):
    def __call__(self, tensor):
        return tensor.to(torch.float32)

In [ ]:
target_size = (930, 930)
pipeline= transforms.Compose([
    #transforms.ToTensor(),  # Convert image to tensor
    transforms.Lambda(lambda x: x[:3]),
    PadToSize(target_size),  # Pad the image to the target size
    ConvertToFloat32()
    #transforms.ToPILImage(), 
])

current means:
tensor([0.2693, 0.1058, 0.3865])

current std_dev:
tensor([0.0505, 0.1627, 0.0643])

eff_dist means:
tensor([0.1720, 0.6108, 0.5125])

eff_dist std_dev:
tensor([0.0962, 0.1022, 0.0644])

pdn_density means:
tensor([0.3741, 0.4506, 0.3820])

pdn_density std_dev:
tensor([0.2636, 0.2681, 0.1267])

ir_drop means:
tensor([0.2233, 0.3428, 0.5156])

ir_drop std_dev:
tensor([0.0592, 0.1559, 0.0490])

In [ ]:
class StackedImagesDataset(Dataset):
    def __init__(self, image_dir, label_dir, transform=None):
        self.image_dir = image_dir
        self.label_dir = label_dir
        self.image_files = sorted(os.listdir(image_dir))  # Sorted to align image order
        self.label_files = sorted(os.listdir(label_dir))  # Assuming labels have the same order
        self.transform = transform
        self.current_transform = transforms.Compose([
            transforms.Normalize(mean=[0.2693, 0.1058, 0.3865], std=[0.0505, 0.1627, 0.0643])  # Normalize
            ])
        self.eff_dist_transform = transforms.Compose([
            transforms.Normalize(mean=[0.1720, 0.6108, 0.5125], std=[0.0962, 0.1022, 0.0644])  # Normalize
            ])
        self.pdn_density_transform = transforms.Compose([
            transforms.Normalize(mean=[0.3741, 0.4506, 0.3820], std=[0.2636, 0.2681, 0.1267])  # Normalize
            ])
        self.ir_drop_transform = transforms.Compose([
            transforms.Normalize(mean=[0.2233, 0.3428, 0.5156], std=[0.0592, 0.1559, 0.0490])  # Normalize
            ])

    
    def __len__(self):
        return len(self.label_files) # based off of the label amount
    
    def __getitem__(self, idx):
        # Get 3 consecutive images for stacking (you could choose any other strategy here)
        names = []

        img1_path = os.path.join(self.image_dir, self.image_files[3*idx])
        img2_path = os.path.join(self.image_dir, self.image_files[3*idx+1])
        img3_path = os.path.join(self.image_dir, self.image_files[3*idx+2])
        
        # Load images
        img1 = read_image(img1_path)
        img2 = read_image(img2_path)
        img3 = read_image(img3_path)
        if self.transform:
            img1 = self.transform(img1)
            img1 = self.current_transform(img1)
            img2 = self.transform(img2)
            img2 = self.eff_dist_transform(img2)
            img3 = self.transform(img3)
            img3 = self.pdn_density_transform(img3)


        # Stack the 3 images along the channel dimension (depth-wise)
        stacked_images = torch.cat([img1,img2,img3])
        
        # Load corresponding label image
        label_path = os.path.join(self.label_dir, self.label_files[idx])  # Label corresponding to last image in stack
        label = read_image(label_path)
        if self.transform:
            label = self.transform(label)  # Apply transformation if needed
            label = self.ir_drop_transform(label)
        
        names.append(self.image_files[3*idx])
        names.append(self.image_files[3*idx+1])
        names.append(self.image_files[3*idx+2])
        names.append(self.label_files[idx])

        return stacked_images, label, names #, idx, names

In [ ]:
image_dataset = StackedImagesDataset(image_dir="/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input", label_dir= "/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/ir_drop", transform=pipeline)

In [ ]:
# for image, label in image_dataset:
#     print(image.size())
#     print(label.size())
#     #print(names)

In [ ]:
train_dataset, test_dataset = torch.utils.data.random_split(image_dataset, [0.8, 0.2])

In [ ]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_dataset, batch_size=5, shuffle=True)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)

In [ ]:
model = vit(image_size = (930,930), patch_size = (15,15), dim = 768, depth = 1, heads = 4, mlp_dim = 10, channels=9, out_channels=3) 

In [ ]:
# model = vit(
#     image_size=(256, 256),  # Input image size
#     patch_size=(16, 16),  # Patch size
#     dim=257,  # Transformer hidden dimension
#     depth=6,  # Number of transformer layers
#     heads=8,  # Number of attention heads
#     mlp_dim=512,  # Feedforward dimension
#     channels=3,  # Input channels (3 for RGB)
#     dropout=0.1,  # Dropout probability
#     emb_dropout=0.1  # Embedding dropout probability
# )

# # Create a random input image (batch_size, channels, height, width)
# input_image = torch.randn(1, 3, 256, 256)

# # Forward pass through the model
# output_image = model(input_image)

# print("Output image shape:", output_image.shape)  # Should be (1, 3, 256, 256)

In [ ]:
# test = torch.rand(1,9,930,930)
# for image, label in image_dataset:
#    output = model(test)
#    print(output.size())
#    break
# only works if the images are batched together

In [ ]:
criterion = nn.L1Loss()  # For multi-class classification
optimizer = optim.Adam(model.parameters(), lr=0.003, weight_decay=1e-5)

# Step 4: Training Loop
epochs = 10  # Number of epochs
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for epoch in range(epochs):
    model.train()  # Set the model to training mode
    running_loss = 0.0

    for batch_idx, (inputs, labels, names) in enumerate(tqdm(train_dataloader, desc=f"Epoch {epoch + 1}", ncols=100)):
        inputs, labels = inputs.to(device), labels.to(device)

        optimizer.zero_grad()  # Zero the gradients before the backward pass
        # print(inputs.size())
        outputs = model(inputs)  # Forward pass
        # print(outputs.size())
        loss = criterion(outputs, labels)  # Calculate the loss
        loss.backward()  # Backward pass
        optimizer.step()  # Optimize the model

        running_loss += loss.item()

    # Print statistics for the current epoch
    print(f"Epoch {epoch+1}/{epochs}")
    print(f"Loss: {running_loss/len(train_dataloader):.4f}")

    # Step 5: Evaluate on Test Data
    model.eval()  # Set the model to evaluation mode
    running_loss = 0.0
    with torch.no_grad():  # No need to track gradients during evaluation
        for batch_idx, (inputs, labels, names) in enumerate(tqdm(test_dataloader, desc=f"Testing Epoch {epoch + 1}", ncols=100)):
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)  # Calculate the loss
            running_loss += loss.item()


    print(f"Test Loss: {running_loss/len(test_dataloader):.4f}")
    print("-" * 50)

print("Training Finished!")

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False)
model.eval()  # Set the model to evaluation mode
running_loss = 0.0
first = 0
with torch.no_grad():  # No need to track gradients during evaluation
    for batch_idx, (inputs, labels, names) in enumerate(tqdm(test_dataloader, desc=f"Testing Epoch {epoch + 1}", ncols=100)):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)  # Calculate the loss
        running_loss += loss.item()
        # print(names)
        if first == 0:
            print(outputs.size())
            outputs = torch.reshape(outputs, (3,930,930))
            image = transforms.ToPILImage()(outputs)
            image.save("inferenced_new_"+names[3][0])
            first = 1


print(f"Test Loss: {running_loss/len(test_dataloader):.4f}")
print("-" * 50)

Without normalization, correction of decoder input and dimension: Loss = 39.4761

temp

In [ ]:
test_dataloader = DataLoader(test_dataset, batch_size=5, shuffle=False)
model.eval()  # Set the model to evaluation mode
running_loss = 0.0
with torch.no_grad():  # No need to track gradients during evaluation
    for batch_idx, (inputs, labels, names) in enumerate(tqdm(test_dataloader, desc=f"Testing Epoch {epoch + 1}", ncols=100)):
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        loss = criterion(outputs, labels)  # Calculate the loss
        running_loss += loss.item()
        print(names)


print(f"Test Loss: {running_loss/len(test_dataloader):.4f}")
print("-" * 50)

In [ ]:
means = [torch.zeros(3),torch.zeros(3),torch.zeros(3),torch.zeros(3)] # current, eff_dist, pdn_density, ir_drop
std_devs = [torch.zeros(3),torch.zeros(3),torch.zeros(3),torch.zeros(3)]
count = [0,0,0,0]
index = 0
directory = os.listdir("/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input")

temp_transform = transforms.Compose([
    transforms.ToTensor()  # Converts to tensor and scales values to [0, 1]
])

for file in directory:
    if "current.png" in file:
        index = 0
    elif "eff_dist.png" in file:
        index = 1
    else: # "pdn_density.png" in file:
        index = 2
    image_path = os.path.join("/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/input", file)
        
    # Open the image
    image = Image.open(image_path).convert('RGB')  # Ensure it's in RGB format
        
    # Apply the transformation
    image_tensor = temp_transform(image)
        
    # Calculate the mean and std for the current image
    means[index] += image_tensor.mean(dim=[1, 2])  # Mean for each channel (R, G, B)
    std_devs[index] += image_tensor.std(dim=[1, 2])  # Std for each channel (R, G, B)
    count[index] += 1

In [ ]:
directory = os.listdir("/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/ir_drop")
for file in directory:
    image_path = os.path.join("/home/bqtx/Documents/VLSI/ir_drop_ml/training_data/png-files/ir_drop", file)
        
    # Open the image
    image = Image.open(image_path).convert('RGB')  # Ensure it's in RGB format
        
    # Apply the transformation
    image_tensor = temp_transform(image)
        
    # Calculate the mean and std for the current image
    means[3] += image_tensor.mean(dim=[1, 2])  # Mean for each channel (R, G, B)
    std_devs[3] += image_tensor.std(dim=[1, 2])  # Std for each channel (R, G, B)
    count[3] += 1

In [ ]:
print("current means:")
print(means[0]/count[0])
print("current std_dev:")
print(std_devs[0]/count[0])
print("eff_dist means:")
print(means[1]/count[1])
print("eff_dist std_dev:")
print(std_devs[1]/count[1])
print("pdn_density means:")
print(means[2]/count[2])
print("pdn_density std_dev:")
print(std_devs[2]/count[2])
print("ir_drop means:")
print(means[3]/count[3])
print("ir_drop std_dev:")
print(std_devs[3]/count[3])